# Feature Engineering — Dropout Risk Prediction

Aggregates early assessments per enrolment (avg_score, pass_rate, assessment_count),
joins enrolment + student attributes. EXCLUDES post-outcome leakage (status, is_completed).

**Reads:** `silver_enrolments`, `silver_students`, `silver_assessments`  **Writes:** `gold_ml_features`

In [ ]:
from pyspark.sql import SparkSession
from pyspark.sql.functions import col, lit, current_timestamp, when, avg, count, round as spark_round

spark = SparkSession.builder.getOrCreate()
print('Spark session ready')

In [ ]:
en = spark.read.table('silver_enrolments')
st = spark.read.table('silver_students')
asm = spark.read.table('silver_assessments')
print(f'enrolments={en.count():,} students={st.count():,} assessments={asm.count():,}')

required = {'enrolment_id', 'student_id', 'is_withdrawn', 'credits'}
missing = required - set(en.columns)
if missing:
    raise ValueError(f'silver_enrolments missing columns {sorted(missing)}. Regenerate data and rerun bronze/silver.')

In [ ]:
# Aggregate assessment performance per enrolment (legitimate early-term features).
asm_agg = (
    asm.groupBy('enrolment_id')
    .agg(
        spark_round(avg('score'), 2).alias('avg_score'),
        spark_round(avg('is_pass'), 4).alias('pass_rate'),
        count('*').alias('assessment_count'),
    )
)

# Join attributes + select pre-outcome features. EXCLUDE leakage (status, is_completed).
ml_features = (
    en.select(
        'enrolment_id', 'student_id', 'department', 'level', 'credits', 'age_at_enrolment',
        col('is_withdrawn').alias('had_dropout'),
    )
    .join(asm_agg, 'enrolment_id', 'left')
    .join(st.select('student_id', 'programme', 'gender', 'region', 'cohort_year'),
          'student_id', 'left')
    .na.fill(0, subset=['avg_score', 'pass_rate', 'assessment_count', 'credits',
                        'age_at_enrolment', 'cohort_year'])
    .na.fill('unknown', subset=['department', 'level', 'programme', 'gender', 'region'])
    .withColumn('feature_timestamp', current_timestamp())
)

total_rows = ml_features.count()
positive_rows = ml_features.filter(col('had_dropout') == 1).count()
dropout_rate = (positive_rows / total_rows * 100) if total_rows else 0.0

if total_rows < 1000 or positive_rows < max(10, int(total_rows * 0.01)):
    raise ValueError(
        f'Label quality check failed: only {positive_rows}/{total_rows} dropout rows '
        f'({dropout_rate:.2f}%). Check is_withdrawn typing and source data.'
    )

ml_features.write.mode('overwrite').option('overwriteSchema', 'true').format('delta').saveAsTable('gold_ml_features')
print(f'Gold ML features written: {total_rows:,} rows | dropout rate {dropout_rate:.1f}%')

In [ ]:
spark.sql('OPTIMIZE gold_ml_features')
print('Feature table optimized')